Task C: Performance Comparison

Prasanna Paithankar (21CS30065)

In [ ]:
!pip install evaluate sentence_transformers sacrebleu rouge_score

In [3]:
import evaluate
from sentence_transformers import SentenceTransformer, util

In [ ]:
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
sim_model = SentenceTransformer("all-MiniLM-L6-v2")


def evaluate(generated_texts, more_preferred_texts, less_preferred_texts):
    results = {}

    refs_more_pref_bleu = [[ref] for ref in more_preferred_texts]
    refs_less_pref_bleu = [[ref] for ref in less_preferred_texts]

    results["bleu_more_preferred"] = bleu_metric.compute(
        predictions=generated_texts, references=refs_more_pref_bleu
    )["score"]
    results["bleu_less_preferred"] = bleu_metric.compute(
        predictions=generated_texts, references=refs_less_pref_bleu
    )["score"]

    results["rougeL_more_preferred"] = rouge_metric.compute(
        predictions=generated_texts, references=more_preferred_texts
    )["rougeL"]
    results["rougeL_less_preferred"] = rouge_metric.compute(
        predictions=generated_texts, references=less_preferred_texts
    )["rougeL"]

    gen_embeddings = sim_model.encode(generated_texts, convert_to_tensor=True)
    more_pref_embeddings = sim_model.encode(
        more_preferred_texts, convert_to_tensor=True
    )
    less_pref_embeddings = sim_model.encode(
        less_preferred_texts, convert_to_tensor=True
    )

    sim_more_preferred = (
        util.cos_sim(gen_embeddings, more_pref_embeddings).diag().mean().item()
    )
    sim_less_preferred = (
        util.cos_sim(gen_embeddings, less_pref_embeddings).diag().mean().item()
    )

    results["semantic_sim_more_preferred"] = sim_more_preferred
    results["semantic_sim_less_preferred"] = sim_less_preferred

    return results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import os

from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, pipeline

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

ds = load_dataset("nrizwan/safe_ai_assignment_1")
df_test = ds["test"].shuffle(seed=42).select(range(500))

In [ ]:
model_name = "gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

ppo_models = [
    "PrasannaPaithankar/gpt2-medium-ppo-full",
    "PrasannaPaithankar/gpt2-medium-ppo-lora",
    "PrasannaPaithankar/gpt2-medium-ppo-qlora",
    "PrasannaPaithankar/gpt2-medium-ppo-prefix",
]
dpo_models = [
    "PrasannaPaithankar/gpt2-medium-dpo-full",
    "PrasannaPaithankar/gpt2-medium-dpo-lora",
    "PrasannaPaithankar/gpt2-medium-dpo-qlora",
    "PrasannaPaithankar/gpt2-medium-dpo-prefix",
]

In [ ]:
for mod in ppo_models:
    pipe = pipeline(
        "text-generation",
        model=mod,
        tokenizer=tokenizer,
        device_map="auto",
        max_new_tokens=256,
    )
    generated_outputs = pipe(
        list(df_test["Question"]),
        batch_size=32,
    )

    generated_texts = [gen[0]["generated_text"] for gen in generated_outputs]

    print(mod + "=========================")
    metrics = evaluate(
        generated_texts, list(df_test["More_Prefered"]), list(df_test["Less_Prefered"])
    )
    print(metrics)
    print()
    print()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

PrasannaPaithankar/gpt2-medium-ppo-full=========================
{'bleu_preferred': 2.3426597543392065, 'bleu_less_preferred': 3.219825103779619, 'rougeL_preferred': np.float64(0.1280838751211574), 'rougeL_less_preferred': np.float64(0.1402720205014092), 'semantic_sim_preferred': 0.5898957848548889, 'semantic_sim_less_preferred': 0.6376482844352722}




adapter_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

ValueError: Could not load model PrasannaPaithankar/gpt2-medium-ppo-prefix with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py", line 232, in load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 374, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4103, in from_pretrained
    loading_info = model.load_adapter(
                   ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/li

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "gpt2-medium"
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto")

adapter_name = "PrasannaPaithankar/gpt2-medium-dpo-prefix"
model = PeftModel.from_pretrained(base_model, adapter_name)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

prompts = list(df_test["Question"])
batch_size = 16
generated_texts = []

with torch.no_grad():
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]

        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(
            model.device
        )

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

        for j, output in enumerate(outputs):
            input_len = inputs.input_ids[j].shape[0]
            new_tokens = output[input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            generated_texts.append(text.strip())

print(adapter_name + "=========================")
metrics = evaluate(
    generated_texts, list(df_test["More_Prefered"]), list(df_test["Less_Prefered"])
)
print(metrics)
print()
print()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

PrasannaPaithankar/gpt2-medium-dpo-prefix=========================
{'bleu_preferred': 0.0008015200068719878, 'bleu_less_preferred': 0.00020241479786475345, 'rougeL_preferred': np.float64(0.06068936214811908), 'rougeL_less_preferred': np.float64(0.053125447979340795), 'semantic_sim_preferred': 0.06575774401426315, 'semantic_sim_less_preferred': 0.04409990832209587}




/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


In [ ]:
for mod in dpo_models:
    pipe = pipeline(
        "text-generation",
        model=mod,
        tokenizer=tokenizer,
        device_map="auto",
        max_new_tokens=256,
    )
    generated_outputs = pipe(
        list(df_test["Question"]),
        batch_size=16,
    )

    generated_texts = [gen[0]["generated_text"] for gen in generated_outputs]

    print(mod + "=========================")
    metrics = evaluate(
        generated_texts, list(df_test["More_Prefered"]), list(df_test["Less_Prefered"])
    )
    print(metrics)
    print()
    print()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

PrasannaPaithankar/gpt2-medium-dpo-full=========================
{'bleu_preferred': 2.5268401158518268, 'bleu_less_preferred': 3.2488010812242964, 'rougeL_preferred': np.float64(0.12958391075443204), 'rougeL_less_preferred': np.float64(0.14140730873742877), 'semantic_sim_preferred': 0.5977009534835815, 'semantic_sim_less_preferred': 0.6423372030258179}




adapter_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

ValueError: Could not load model PrasannaPaithankar/gpt2-medium-dpo-prefix with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py", line 232, in load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 374, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4103, in from_pretrained
    loading_info = model.load_adapter(
                   ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/li

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "gpt2-medium"
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto")

adapter_name = "PrasannaPaithankar/gpt2-medium-ppo-prefix"
model = PeftModel.from_pretrained(base_model, adapter_name)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

prompts = list(df_test["Question"])
batch_size = 16
generated_texts = []

with torch.no_grad():
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]

        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(
            model.device
        )

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

        for j, output in enumerate(outputs):
            input_len = inputs.input_ids[j].shape[0]
            new_tokens = output[input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            generated_texts.append(text.strip())

print(adapter_name + "=========================")
metrics = evaluate(
    generated_texts, list(df_test["More_Prefered"]), list(df_test["Less_Prefered"])
)
print(metrics)
print()
print()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Starting generation...
PrasannaPaithankar/gpt2-medium-ppo-prefix=========================
{'bleu_preferred': 0.0026507753717888685, 'bleu_less_preferred': 0.00039513412578094534, 'rougeL_preferred': np.float64(0.06902543014371133), 'rougeL_less_preferred': np.float64(0.05745879918371958), 'semantic_sim_preferred': 0.030969899147748947, 'semantic_sim_less_preferred': 0.011710470542311668}


